In [1]:
# Install stable compatible versions and evaluation metrics
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers[torch] datasets accelerate evaluate rouge_score bert-score --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 113.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 105.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.4 MB/s eta 0:00:0000:0100:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 M

In [2]:
# 1) Imports and dataset loading
from datasets import load_dataset
from transformers import AutoTokenizer
import os

# Load the CLEANED UPDATED dataset (~2% refusals preserved)
candidate_dataset_paths = [
    'curate_dataset_updated_clean.csv',
    'backend/curate_dataset_updated_clean.csv',
    '/content/curate_dataset_updated_clean.csv',
    '/content/drive/MyDrive/curate_dataset_updated_clean.csv',
]
dataset_path = next((p for p in candidate_dataset_paths if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError('Could not find curate_dataset_updated_clean.csv. Run clean_dataset.ipynb first.')
print('Loading dataset from:', dataset_path)
dataset = load_dataset('csv', data_files=dataset_path)
print(dataset)
print('Columns:', dataset['train'].column_names)
print(f'Total rows: {len(dataset["train"])}')
print('Sample row:')
print(dataset['train'][0])

Loading dataset from: curate_dataset_updated_clean.csv


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['fact_id', 'context', 'question', 'answer'],
        num_rows: 2870
    })
})
Columns: ['fact_id', 'context', 'question', 'answer']
Total rows: 2870
Sample row:
{'fact_id': 'themeeting_commission', 'context': 'Created between 1771 and 1773, Jean-Honoré Fragonard’s The Meeting is the second movement in The Progress of Love cycle, commissioned by Madame du Barry for her Louveciennes pavilion. At this career zenith, Fragonard moved away from rigid history painting to embrace the hedonistic, private patronage of the French aristocracy. The composition utilizes a theatrical, V-shaped structure where a young suitor scales a garden wall to reach his lover, their surprised, stagelike poses suggesting a clandestine rendezvous inspired by contemporary ballet. Beneath a towering statue of Venus and Cupid—the former seizing the latter’s arrows to symbolize love’s power—the couple is enveloped by an unruly, surging garden. Fragonard’s technique 

In [3]:
# 2) Preprocess: build prompt and target

def make_example(ex):
    question = ex.get('question', '').strip()
    context = ex.get('context', '').strip()
    answer = ex.get('answer', '').strip()
    fact_id = ex.get('fact_id', '')  # Preserve for stratified splitting
    input_text = f"Question: {question}\nContext: {context}\nAnswer:"
    return {'input_text': input_text, 'target_text': answer, 'fact_id': fact_id}

# Map dataset
mapped = dataset['train'].map(lambda x: make_example(x))

# Deduplicate on exact (input_text, target_text) pairs only
seen = set()
deduplicated = []
for ex in mapped:
    key = (ex['input_text'].strip(), ex['target_text'].strip())
    if key not in seen:
        seen.add(key)
        deduplicated.append(ex)

print(f'Original: {len(mapped)}, After exact-dedup: {len(deduplicated)}')
print(f'Rows removed: {len(mapped) - len(deduplicated)}')
print('Sample:', deduplicated[0])

Map:   0%|          | 0/2870 [00:00<?, ? examples/s]

Original: 2870, After exact-dedup: 2870
Rows removed: 0
Sample: {'fact_id': 'themeeting_commission', 'context': 'Created between 1771 and 1773, Jean-Honoré Fragonard’s The Meeting is the second movement in The Progress of Love cycle, commissioned by Madame du Barry for her Louveciennes pavilion. At this career zenith, Fragonard moved away from rigid history painting to embrace the hedonistic, private patronage of the French aristocracy. The composition utilizes a theatrical, V-shaped structure where a young suitor scales a garden wall to reach his lover, their surprised, stagelike poses suggesting a clandestine rendezvous inspired by contemporary ballet. Beneath a towering statue of Venus and Cupid—the former seizing the latter’s arrows to symbolize love’s power—the couple is enveloped by an unruly, surging garden. Fragonard’s technique features fluid, feathery brushwork and a luminous pastel palette, characterizing the peak of the Rococo style. The dense, vertiginous foliage serves as

In [4]:
# 3) Tokenization function
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

max_input_length = 512
max_target_length = 96

def tokenize_fn(examples):
    inputs = examples['input_text']
    targets = examples['target_text']
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    label_ids = labels['input_ids']
    label_ids = [[(l if l != tokenizer.pad_token_id else -100) for l in lab] for lab in label_ids]
    model_inputs['labels'] = label_ids
    return model_inputs

# Convert to Dataset and tokenize
deduplicated_dataset = Dataset.from_dict({k: [ex[k] for ex in deduplicated] for k in ['input_text', 'target_text', 'fact_id']})
tokenized = deduplicated_dataset.map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])
print('Tokenized dataset:', tokenized)
print('Sample fact_id:', tokenized['fact_id'][:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2870 [00:00<?, ? examples/s]

Tokenized dataset: Dataset({
    features: ['fact_id', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2870
})
Sample fact_id: ['themeeting_commission', 'womancleaningturnips_capacity', 'monalisa_view', 'monalisa_renaissance', 'libertyleading_style']


In [5]:
# 4) Train / validation split with stratification by fact_id (prevents data leakage)
from sklearn.model_selection import train_test_split
from collections import Counter

# Stratify by fact_id prefix to group all questions about same artwork together
fact_ids = tokenized['fact_id']
fact_id_prefixes = [fid.rsplit('_', 1)[0] if '_' in fid else fid for fid in fact_ids]

# Check stratification feasibility
prefix_counts = Counter(fact_id_prefixes)
stratifiable = all(count >= 2 for count in prefix_counts.values())

indices = list(range(len(tokenized)))

if stratifiable:
    train_indices, eval_indices = train_test_split(
        indices,
        test_size=0.05,
        random_state=42,
        stratify=fact_id_prefixes
    )
    print("✓ Using stratified split by artwork (prevents data leakage)")
else:
    train_indices, eval_indices = train_test_split(
        indices,
        test_size=0.05,
        random_state=42
    )
    print("⚠️  Stratification not feasible, using random split")

train_ds = tokenized.select(train_indices)
eval_ds = tokenized.select(eval_indices)

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')
print(f'Train artworks (sample): {dict(Counter(train_ds["fact_id"]).most_common(5))}')
print(f'Eval artworks (sample): {dict(Counter(eval_ds["fact_id"]).most_common(5))}')
print('✓ Stratified split complete: all artwork questions stay together')

✓ Using stratified split by artwork (prevents data leakage)
Train: 2726 | Eval: 144
Train artworks (sample): {'global_docent_fallback': 53, 'womancleaningturnips_palette': 8, 'persistence_soft': 8, 'creationofadam_strain': 8, 'womancleaningturnips_title': 8}
Eval artworks (sample): {'global_docent_fallback': 3, 'ladyermine_reference': 2, 'ladyermine_transition': 2, 'persistence_medium': 2, 'libertyleading_event': 2}
✓ Stratified split complete: all artwork questions stay together


In [6]:
# 5) Prepare trainer and model
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch
import gc

# Clear GPU cache before loading model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

output_dir = '/content/drive/MyDrive/flan_t5_base_docent_updated_clean' if os.path.exists('/content/drive/MyDrive') else './flan_t5_base_docent_updated_clean'

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    predict_with_generate=True,
    eval_strategy='steps',
    eval_steps=100,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.10,
    num_train_epochs=6,
    label_smoothing_factor=0.05,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    weight_decay=0.01,
    report_to='none',
    generation_num_beams=4,
    generation_max_length=max_target_length,
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
# 6) Initialize Trainer and train
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Start training
trainer.train()

Step,Training Loss,Validation Loss
100,3.504337,1.545739
200,2.741763,1.272964
300,2.504335,1.161385
400,2.328563,1.103681
500,2.268697,1.064042
600,2.178265,1.034294
700,2.089797,1.016387
800,2.107362,1.009956
900,2.097246,1.005012
1000,2.098515,1.005460


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1026, training_loss=2.5211954119145057, metrics={'train_runtime': 2816.2051, 'train_samples_per_second': 5.808, 'train_steps_per_second': 0.364, 'total_flos': 8816785879449600.0, 'train_loss': 2.5211954119145057, 'epoch': 6.0})

In [8]:
# 7) Save the fine-tuned model
save_path = output_dir + '/final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('Saved to:', save_path)
print('✅ Training completed! Model saved.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./flan_t5_base_docent_updated_clean/final
✅ Training completed! Model saved.


In [9]:
# 8) Verify save
if os.path.exists(save_path):
    print(f"Contents of {save_path}:")
    print(os.listdir(save_path))
else:
    print(f"Directory not found: {save_path}")

Contents of ./flan_t5_base_docent_updated_clean/final:
['training_args.bin', 'config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']


In [10]:
# 9) Create zip for download
import shutil

zip_name = '/content/flan_t5_base_docent_updated_clean_final.zip'
zip_base = zip_name.replace('.zip', '')

if os.path.exists(save_path):
    if os.path.exists(zip_name):
        os.remove(zip_name)
    shutil.make_archive(zip_base, 'zip', save_path)
    print('Created zip:', zip_name)
else:
    print(f'Cannot create zip; save_path not found: {save_path}')

Created zip: /content/flan_t5_base_docent_updated_clean_final.zip


In [11]:
# 10) Download from Colab or copy to Drive
try:
    from google.colab import files
    if os.path.exists(zip_name):
        print(f'Downloading: {zip_name}')
        files.download(zip_name)
    else:
        print(f'File not found: {zip_name}')
except ImportError:
    print('Not in Colab environment (or google.colab not available)')

Downloading: /content/flan_t5_base_docent_updated_clean_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# 11) Inference and evaluation
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate
import torch
import os

print("--- Loading baseline model for inference (flan-t5-base on uncleaned data) ---")

# Load tokenizer and model directly from the saved path.
if not os.path.exists(save_path):
    raise FileNotFoundError(f"save_path not found: {save_path}. Run the save cell first.")

tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)
model.config.tie_word_embeddings = False
model.to('cuda' if torch.cuda.is_available() else 'cpu')


def generate_answer(question, context, max_new_tokens=96):
    prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        length_penalty=1.1,
        early_stopping=True,
        do_sample=False,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def parse_input_text_for_qa(decoded_text):
    """Parse question and context from decoded text. Handles missing newlines."""
    question_prefix = "Question:"
    context_prefix = "Context:"
    answer_prefix = "Answer:"
    
    question_text = ""
    context_text = ""
    
    q_idx = decoded_text.find(question_prefix)
    if q_idx != -1:
        q_start = q_idx + len(question_prefix)
        c_idx = decoded_text.find(context_prefix, q_start)
        if c_idx != -1:
            question_text = decoded_text[q_start:c_idx].strip()
            c_start = c_idx + len(context_prefix)
            a_idx = decoded_text.find(answer_prefix, c_start)
            if a_idx != -1:
                context_text = decoded_text[c_start:a_idx].strip()
            else:
                context_text = decoded_text[c_start:].strip()
    
    return question_text, context_text


print("--- Running Inference Samples (UPDATED model on cleaned data) ---")
print("⚠️  Expect repetitive or generic outputs due to fallback answer contamination\n")
for i in range(3):
    ex = eval_ds[i]
    input_text = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    target_text = tokenizer.decode([l for l in ex['labels'] if l != -100], skip_special_tokens=True)
    
    q_part, c_part = parse_input_text_for_qa(input_text)
    
    print(f"\n=== Sample {i+1} ===")
    print(f"Question: {q_part[:60] if q_part else '(EMPTY)'}...")
    print(f"Context: {c_part[:60] if c_part else '(EMPTY)'}...")
    
    pred = generate_answer(q_part, c_part)
    
    print(f"Pred: {pred[:80]}...")
    print(f"Ref: {target_text[:80]}...")
    print('---')

# Compute ROUGE and BERTScore
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
preds = []
refs = []
print("\nComputing ROUGE/BERTScore for BASELINE eval set...")
for ex in eval_ds:
    input_text = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    target_text = tokenizer.decode([l for l in ex['labels'] if l != -100], skip_special_tokens=True)
    
    q_part, c_part = parse_input_text_for_qa(input_text)
    preds.append(generate_answer(q_part, c_part))
    refs.append(target_text)

rouge_results = rouge.compute(predictions=preds, references=refs)
bertscore_results = bertscore.compute(predictions=preds, references=refs, lang='en')
bertscore_precision = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
bertscore_recall = sum(bertscore_results['recall']) / len(bertscore_results['recall'])
bertscore_f1 = sum(bertscore_results['f1']) / len(bertscore_results['f1'])

print('\n=== METRICS (Cleaned Dataset) ===')
print('ROUGE Results:', rouge_results)
print(f'BERTScore Precision: {bertscore_precision:.4f}')
print(f'BERTScore Recall: {bertscore_recall:.4f}')
print(f'BERTScore F1: {bertscore_f1:.4f}')
print('Eval loss:', trainer.evaluate().get('eval_loss'))

--- Loading baseline model for inference (flan-t5-base on uncleaned data) ---


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- Running Inference Samples (UPDATED model on cleaned data) ---
⚠️  Expect repetitive or generic outputs due to fallback answer contamination


=== Sample 1 ===
Question: What is the connection to the Order of the Ermine?...
Context: Painted around 1489–1491 while Leonardo da Vinci was the cou...
Pred: It also alludes to Cecilia’s own surname....
Ref: Historically, it refers to Ludovico, who was a member of the Order of the Ermine...
---

=== Sample 2 ===
Question: Is the ermine's paw accurate?...
Context: Painted around 1489–1491 while Leonardo da Vinci was the cou...
Pred: Furthermore, his anatomical precision shows in the muscular detail of the ermine...
Ref: Furthermore, his anatomical precision shows in the muscular detail of the ermine...
---

=== Sample 3 ===
Question: What was the artist thinking when he made this?...
Context: Eugène Delacroix painted Liberty Leading the People in the a...
Pred: Ultimately, this monumental work emerged during a period of personal conviction .

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== METRICS (Cleaned Dataset) ===
ROUGE Results: {'rouge1': np.float64(0.7541459861844322), 'rouge2': np.float64(0.6951421414285531), 'rougeL': np.float64(0.7495172481747243), 'rougeLsum': np.float64(0.7492315171269359)}
BERTScore Precision: 0.9654
BERTScore Recall: 0.9644
BERTScore F1: 0.9648


Training Loss,Validation Loss,Step
2.114103,1.005012,1026


Eval loss: 1.0050122737884521


In [18]:
# 12) Evaluation metrics
import math
import re
from collections import Counter
import evaluate


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match(prediction, reference):
    return int(normalize_answer(prediction) == normalize_answer(reference))


def token_prf(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0, 0.0

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    overlap = sum((pred_counts & ref_counts).values())
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)
    return precision, recall, f1


def parse_question_context(example, tokenizer):
    if 'input_text' in example and str(example.get('input_text', '')).strip():
        decoded_text = str(example['input_text'])
    elif 'input_ids' in example and example['input_ids']:
        decoded_text = tokenizer.decode(example['input_ids'], skip_special_tokens=True)
    else:
        decoded_text = ''

    question = ''
    context = ''
    question_prefix = 'Question:'
    context_prefix = 'Context:'
    answer_prefix = 'Answer:'

    q_start = decoded_text.find(question_prefix)
    if q_start != -1:
        q_start += len(question_prefix)
        c_start = decoded_text.find(context_prefix, q_start)
        if c_start != -1:
            question = decoded_text[q_start:c_start].strip()
            c_start += len(context_prefix)
            a_start = decoded_text.find(answer_prefix, c_start)
            if a_start != -1:
                context = decoded_text[c_start:a_start].strip()
            else:
                context = decoded_text[c_start:].strip()

    return question, context


def decode_reference(example, tokenizer):
    if 'target_text' in example and str(example['target_text']).strip():
        return str(example['target_text']).strip()
    labels = example.get('labels', [])
    if labels:
        label_ids = [token_id for token_id in labels if token_id != -100]
        if label_ids:
            return tokenizer.decode(label_ids, skip_special_tokens=True).strip()
    return ''


# Evaluate on a manageable subset for quick Colab feedback.
metric_eval_size = min(100, len(eval_ds))
metric_eval_ds = eval_ds.select(range(metric_eval_size))
metric_preds = []
metric_refs = []

for example in metric_eval_ds:
    question, context = parse_question_context(example, tokenizer)
    prediction = generate_answer(question, context)
    reference = decode_reference(example, tokenizer)
    if reference:
        metric_preds.append(prediction)
        metric_refs.append(reference)

if not metric_refs:
    raise ValueError(
        'No references were found in the selected eval subset. Check the labels or increase the metric_eval_size.'
    )

# Accuracy / exact match and token-level precision, recall, F1
ems = [exact_match(pred, ref) for pred, ref in zip(metric_preds, metric_refs)]
prfs = [token_prf(pred, ref) for pred, ref in zip(metric_preds, metric_refs)]
avg_precision = sum(p for p, _, _ in prfs) / len(prfs)
avg_recall = sum(r for _, r, _ in prfs) / len(prfs)
avg_f1 = sum(f for _, _, f in prfs) / len(prfs)
accuracy = sum(ems) / len(ems)

# ROUGE
rouge = evaluate.load('rouge')
rouge_scores = rouge.compute(predictions=metric_preds, references=metric_refs)

# BERTScore: use an explicit uncased BERT backbone
bertscore = evaluate.load('bertscore')
bertscore_scores = bertscore.compute(
    predictions=metric_preds,
    references=metric_refs,
    lang='en',
    model_type='bert-base-uncased',
)
bertscore_precision = sum(bertscore_scores['precision']) / len(bertscore_scores['precision'])
bertscore_recall = sum(bertscore_scores['recall']) / len(bertscore_scores['recall'])
bertscore_f1 = sum(bertscore_scores['f1']) / len(bertscore_scores['f1'])

# Perplexity from eval loss
trainer_eval = trainer.evaluate()
perplexity = None
if 'eval_loss' in trainer_eval and trainer_eval['eval_loss'] is not None:
    perplexity = math.exp(trainer_eval['eval_loss'])

print('\n=== METRICS (100-sample subset on cleaned dataset) ===')
print('Metric subset size:', metric_eval_size)
print('Accuracy / Exact Match:', accuracy)
print('Precision:', avg_precision)
print('Recall:', avg_recall)
print('F1:', avg_f1)
print('ROUGE:', rouge_scores)
print('BERTScore Precision:', bertscore_precision)
print('BERTScore Recall:', bertscore_recall)
print('BERTScore F1:', bertscore_f1)
print('Eval loss:', trainer_eval.get('eval_loss'))
print('Perplexity:', perplexity)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training Loss,Validation Loss,Step
2.114103,1.005012,1026



=== METRICS (100-sample subset on cleaned dataset) ===
Metric subset size: 100
Accuracy / Exact Match: 0.42
Precision: 0.7812286659419013
Recall: 0.7480499761676233
F1: 0.7602543711043063
ROUGE: {'rouge1': np.float64(0.7699294714801683), 'rouge2': np.float64(0.7134483284899504), 'rougeL': np.float64(0.767205434708359), 'rougeLsum': np.float64(0.7645475040603423)}
BERTScore Precision: 0.8726443487405777
BERTScore Recall: 0.8580200046300888
BERTScore F1: 0.8641418150067329
Eval loss: 1.0050122737884521
Perplexity: 2.731940803883641


In [19]:
import os
import shutil
from google.colab import drive

if 'zip_name' not in globals():
    zip_name = '/content/flan_t5_base_docent_updated_clean_final.zip'

if not os.path.exists(zip_name):
    print(f'Zip not found: {zip_name}. Run the zip creation cell first.')
else:
    drive_root = '/content/drive'
    my_drive = '/content/drive/MyDrive'

    if not os.path.exists(my_drive):
        drive.mount(drive_root, force_remount=True)

    drive_zip = os.path.join(my_drive, os.path.basename(zip_name))
    shutil.copy2(zip_name, drive_zip)
    print(f'Copied updated zip to Google Drive: {drive_zip}')

    if os.path.exists(drive_zip):
        print(f'Verified: {drive_zip}')
    else:
        print('Copy verification failed.')

Mounted at /content/drive
Copied updated zip to Google Drive: /content/drive/MyDrive/flan_t5_base_docent_updated_clean_final.zip
Verified: /content/drive/MyDrive/flan_t5_base_docent_updated_clean_final.zip
